# Module 2 — Filtering, Cloud Masking & Compositing

**Goal:** turn a noisy, cloud-contaminated `ImageCollection` into a clean analysis-ready dataset. Learn `.map()`, which is the single most important method in the whole API.

**Time:** ~1.5–2 hours

### Why this matters for the glyphosate project
Raw satellite time series are full of clouds, shadows, and sensor artifacts. Before you can build a harmonic baseline or detect a spray-induced dip in vegetation index, you need every image in the stack cleaned the same way. That cleaning step is always written as a function you `.map()` over the collection — this is the pattern you'll reuse for computing NDVI across a whole year of imagery.


In [ ]:
import ee
try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')


### `.map()` — apply a function to every image, server-side

You write an ordinary-looking Python function that takes one `ee.Image` and returns one `ee.Image`. GEE ships that function to the server and runs it across the whole collection in parallel. Two hard rules:
- No Python `if`/`for` on server-side values inside the function — use `ee.Algorithms`/image methods instead (e.g. `.where()`, boolean masks) since the function body runs *symbolically*, not with real values in hand.
- The function must not call `.getInfo()` or anything else that forces evaluation — that would mean pulling every image down to Python one at a time, defeating the entire point.


In [ ]:
# Sentinel-2 surface reflectance ships a per-pixel cloud probability band
# in a companion collection, and a simpler QA60 bitmask band on the image itself.

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate('2021-01-01', '2021-12-31')
      .filterBounds(ee.Geometry.Point(-121.5, 38.5))
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
      .map(mask_s2_clouds))

print('Images after filtering:', s2.size().getInfo())


Notice `.updateMask()` — it doesn't delete pixels, it marks them as masked/nodata. Masked pixels are automatically excluded from reducers later (Module 4), which is exactly the behavior you want: cloudy pixels shouldn't corrupt a vegetation index time series.

### Exercise 2.1
Write a cloud mask function for Landsat 8/9 Collection 2 Level 2 using the `QA_PIXEL` band (bits 3 = cloud shadow, 4 = cloud). Apply it with `.map()` to a year of `LANDSAT/LC08/C02/T1_L2` over a point of your choice, and print the collection size before and after the cloudy-scene filter.


In [ ]:
# TODO: your code here


### Compositing — collapsing many images into one

Sometimes you want a single "best pixel" image out of a noisy stack (e.g. a cloud-free mosaic for a quarter). `.median()`, `.mean()`, `.mosaic()`, and `.qualityMosaic()` all reduce an `ImageCollection` down to one `ee.Image`, per-pixel, ignoring masked values.

In [ ]:
composite = s2.median()
print(composite.bandNames().getInfo())


### Exercise 2.2
Build a leaf-on (e.g. June–August) and leaf-off (e.g. December–February) median composite for the same location, both from your cleaned `s2` collection filtered to different date ranges. This "seasonal composite" pattern is exactly how you'd build a stable baseline image to compare a given week against.

In [ ]:
# TODO: your code here


### Project 2 — "Cloud-free NDVI mosaic builder"

Write a function `seasonal_ndvi_composite(aoi, start, end, max_cloud_pct=40)` that:
1. Filters Sentinel-2 SR to `aoi` (an `ee.Geometry`) and the date range
2. Filters out scenes above `max_cloud_pct` cloud cover
3. Masks clouds per-pixel with `.map()`
4. Computes NDVI per image with `.map()` (add it as a band, don't discard the rest)
5. Returns the **median composite** `ee.Image`, with just the NDVI band selected

Then use `geemap` (or `.getThumbURL()`) to visualize the result over a region in California's Central Valley.


In [ ]:
def seasonal_ndvi_composite(aoi, start, end, max_cloud_pct=40):
    # TODO
    pass

# Visualize:
# import geemap
# Map = geemap.Map()
# aoi = ee.Geometry.Rectangle([-121.8, 38.2, -121.2, 38.8])
# ndvi_img = seasonal_ndvi_composite(aoi, '2021-06-01', '2021-08-31')
# Map.addLayer(ndvi_img, {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}, 'NDVI')
# Map.centerObject(aoi, 9)
# Map


---
**Checkpoint:** you should be comfortable writing a `.map()` function that cleans + derives a band per-image, and you should understand why `.median()`/`.mosaic()` collapse a noisy stack into one usable image. Next module: turning collections into actual *time series* at a point or region, which is the core operation behind the glyphosate change-detection idea.